In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACIÓN DEL DRIVER ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES ---
datos_totales = []
objetivo_registros = 100

# Múltiples URLs (clave para llegar al objetivo)
urls = [
    "https://www.pisos.com/venta/pisos-madrid/",
    "https://www.pisos.com/venta/pisos-barcelona/",
    "https://www.pisos.com/venta/pisos-valencia/",
    "https://www.pisos.com/venta/pisos-sevilla/"
]

try:
    # --- PASO 3: RECORRIDO MULTI-FUENTE ---
    for url in urls:
        if len(datos_totales) >= objetivo_registros:
            break

        print(f"\nEntrando a: {url}")
        driver.get(url)
        pagina = 1

        while len(datos_totales) < objetivo_registros:
            print(f"--- Procesando Página {pagina} ---")

            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "ad-preview"))
                )
            except:
                print("No cargaron propiedades.")
                break

            propiedades = driver.find_elements(By.CLASS_NAME, "ad-preview")

            for p in propiedades:
                datos_totales.append(p.text)

                # Corte inmediato al llegar a 100
                if len(datos_totales) >= objetivo_registros:
                    break

            print(f"Registros acumulados: {len(datos_totales)}")

            # Intentar siguiente página
            try:
                boton_siguiente = driver.find_element(
                    By.XPATH, "//a[contains(@class,'next') or contains(.,'Siguiente')]"
                )
                driver.execute_script("arguments[0].click();", boton_siguiente)
                time.sleep(3)
                pagina += 1

            except:
                print("No hay más páginas en esta URL.")
                break

    print("\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error durante la ejecución: {e}")

finally:
    # --- PASO 4: CIERRE SEGURO ---
    driver.quit()


Entrando a: https://www.pisos.com/venta/pisos-madrid/
--- Procesando Página 1 ---
Registros acumulados: 30
--- Procesando Página 2 ---
Registros acumulados: 60
--- Procesando Página 3 ---
Registros acumulados: 90
--- Procesando Página 4 ---
Registros acumulados: 100

Proceso finalizado con éxito.
Total de registros capturados: 100
